In [1]:
import GridMLM_tokenizers
# from GridMLM_tokenizers import get_chord_melody_features
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from graph_utils import append_graph_ready_object_to_dataset, make_graph_ready_for_dataset_item
import pickle

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
chord_features = GridMLM_tokenizers.CHORD_FEATURES
chord_id_features = {tokenizer.vocab[k]: v for k, v in chord_features.items()}
print(chord_id_features)

{7: {'quality': 'maj', 'root': 0, 'pitch_classes': [0, 4, 7]}, 36: {'quality': 'maj', 'root': 1, 'pitch_classes': [1, 5, 8]}, 65: {'quality': 'maj', 'root': 2, 'pitch_classes': [2, 6, 9]}, 94: {'quality': 'maj', 'root': 3, 'pitch_classes': [3, 7, 10]}, 123: {'quality': 'maj', 'root': 4, 'pitch_classes': [4, 8, 11]}, 152: {'quality': 'maj', 'root': 5, 'pitch_classes': [0, 5, 9]}, 181: {'quality': 'maj', 'root': 6, 'pitch_classes': [1, 6, 10]}, 210: {'quality': 'maj', 'root': 7, 'pitch_classes': [2, 7, 11]}, 239: {'quality': 'maj', 'root': 8, 'pitch_classes': [0, 3, 8]}, 268: {'quality': 'maj', 'root': 9, 'pitch_classes': [1, 4, 9]}, 297: {'quality': 'maj', 'root': 10, 'pitch_classes': [2, 5, 10]}, 326: {'quality': 'maj', 'root': 11, 'pitch_classes': [3, 6, 11]}, 8: {'quality': 'min', 'root': 0, 'pitch_classes': [0, 3, 7]}, 37: {'quality': 'min', 'root': 1, 'pitch_classes': [1, 4, 8]}, 66: {'quality': 'min', 'root': 2, 'pitch_classes': [2, 5, 9]}, 95: {'quality': 'min', 'root': 3, 'pitch

In [4]:
print(chord_features['F:maj7'])
print(chord_id_features[159])
# cf0 = get_chord_pitch_features(chord_features['D:maj7'])
# print(cf0)

{'quality': 'maj7', 'root': 5, 'pitch_classes': [0, 4, 5, 9]}
{'quality': 'maj7', 'root': 5, 'pitch_classes': [0, 4, 5, 9]}


In [5]:
train_gjt = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_train'

In [6]:
train_dataset_gjt = CSGridMLMDataset(train_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')

Loading data file.


In [7]:
d = train_dataset_gjt[0]

In [8]:
graph_ready_objects = make_graph_ready_for_dataset_item(d, tokenizer)

In [9]:
print(d['harmony_ids'])

[6, 276, 276, 194, 194, 6, 339, 339, 148, 148, 6, 276, 276, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 159, 159, 6, 332, 332, 129, 129, 6, 274, 274, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 332, 332, 6, 339, 339, 148, 148, 6, 279, 279, 279, 279, 6, 158, 158, 148, 148, 6, 270, 270, 270, 270, 6, 73, 73, 245, 245, 6, 227, 227, 226, 226, 6, 14, 14, 151, 151]


In [10]:
graph_ready_objects.print_info()

Number of bars: 16
No segment graph created yet.
Bar 1:
Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0., 0., 1., 0., 0., 1., 1., 0.],
        [0., 0., 0., 1., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 1., 0., 0.]])
Bar 2:
Bar token positions: [6, 7, 8, 9]
Number of chord objects in bar: 2
Chord object 1:
Chord lab

In [11]:
print(tokenizer.ids_to_tokens[332])
print(chord_id_features[332])

B:7
{'quality': '7', 'root': 11, 'pitch_classes': [3, 6, 9, 11]}


In [12]:
gjt_train = append_graph_ready_object_to_dataset(train_dataset_gjt)

In [13]:
import os
os.makedirs('data', exist_ok=True)
with open('data/gjt_train.pkl', 'wb') as f:
    pickle.dump(gjt_train, f)

In [14]:
print(gjt_train[0].keys())
# chord_objects = gjt_train[0]['chord_objects']
# chord_objects[5][0].print_info()

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'graph_ready_object'])


In [15]:
d = gjt_train[0]
d['graph_ready_object'].make_graph_of_segment(0, 4)

In [16]:
d['graph_ready_object'].print_info(print_graph=True, print_bars=False)

Number of bars: 16
Segment bar range: [0, 4)
Segment graph features:
HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[8, 1] },
  (pitch, participates, event)={
    edge_index=[2, 38],
    edge_attr=[38, 8],
  },
  (event, next, event)={
    edge_index=[2, 7],
    edge_attr=[7, 7],
  }
)
Segment graph bars:
Bar 1:
Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0

In [17]:
g = d['graph_ready_object'].segment_graph
print(g)
print(g["pitch", "participates", "event"].edge_index)
print(g['event', 'next', 'event'].edge_attr)

HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[8, 1] },
  (pitch, participates, event)={
    edge_index=[2, 38],
    edge_attr=[38, 8],
  },
  (event, next, event)={
    edge_index=[2, 7],
    edge_attr=[7, 7],
  }
)
tensor([[ 0,  4,  7,  9, 11,  0,  4,  6,  9,  2,  5,  9, 11,  0,  2,  4,  5,  8,
         11,  0,  4,  7,  9,  2,  0,  2,  6,  9,  4,  2,  5,  7, 11,  4,  0,  4,
          7, 10],
        [ 0,  0,  0,  0,  0,  1,  1,  1,  1,  2,  2,  2,  2,  2,  3,  3,  3,  3,
          3,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,  6,  6,  7,  7,
          7,  7]])
tensor([[2.0000, 1.0000, 0.0000, 0.5000, 0.0000, 1.0000, 0.0000],
        [2.0000, 0.0000, 0.0000, 0.2857, 0.0000, 1.0000, 1.0000],
        [2.0000, 1.0000, 0.0000, 0.4286, 0.0000, 1.0000, 1.0000],
        [2.0000, 1.0000, 0.0000, 0.2500, 1.0000, 0.0000, 1.0000],
        [2.0000, 1.0000, 1.0000, 0.6667, 0.0000, 0.0000, 1.0000],
        [2.0000, 1.0000, 0.0000, 0.2500, 1.0000, 0.0000, 1.0000],
        [2.0000, 1.0000,

In [20]:
for i in range(4):
    d['graph_ready_object'].print_info(print_graph=True, print_bars=False)

Number of bars: 16
Segment bar range: [0, 4)
Segment graph features:
HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[8, 1] },
  (pitch, participates, event)={
    edge_index=[2, 38],
    edge_attr=[38, 8],
  },
  (event, next, event)={
    edge_index=[2, 7],
    edge_attr=[7, 7],
  }
)
Segment graph bars:
Bar 1:
Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0